# Lab 4 — U-Net on the Semantic Drone Dataset (Google Colab)

**Before you start:** go to **Runtime → Change runtime type → T4 GPU → Save**.

You need `colab_upload.zip` from your lab folder in **Google Drive → My Drive → Lab4/**.
It has `unet_lab.py` plus the pre-resized dataset (`outputs/cache_*.npz`, ~105 MB), so you **don't** need to upload the 4 GB of images.
Optional: also put `best_unet.pth` (the provided pretrained model) in the same Drive folder.

## 1. Check the GPU

In [ ]:
!nvidia-smi -L
import torch; print('CUDA available:', torch.cuda.is_available())

## 2. Mount Google Drive and unpack the lab files

In [ ]:
import glob, os
from google.colab import drive
drive.mount('/content/drive')

# Find colab_upload.zip anywhere in your Drive (also matches 'colab_upload (1).zip' etc.)
hits = sorted(glob.glob('/content/drive/MyDrive/**/colab_upload*.zip', recursive=True))
if hits:
    ZIP = hits[0]
    print('Found zip in Drive:', ZIP)
else:
    # Fallback: upload it straight from your PC (choose colab_upload.zip from your Lab 4 folder)
    print('colab_upload.zip not found in Drive -> please choose it in the upload box below')
    from google.colab import files
    %cd /content
    up = files.upload()
    ZIP = os.path.abspath(next(iter(up)))   # saved in the current folder
DRIVE_DIR = os.path.dirname(ZIP) if ZIP.startswith('/content/drive') else '/content/drive/MyDrive/Lab4'
os.makedirs(DRIVE_DIR, exist_ok=True)   # results get saved here in step 5

!mkdir -p /content/lab4
!unzip -oq "$ZIP" -d /content/lab4
pth = sorted(glob.glob('/content/drive/MyDrive/**/best_unet.pth', recursive=True))
if pth:
    !cp -n "{pth[0]}" /content/lab4/
else:
    print('(no best_unet.pth in Drive - fine for training)')
%cd /content/lab4
!ls -la . outputs


## 3. Train — Tasks 1, 2, 3 and 4
Adam lr=1e-3, 10 epochs (Task 3 asks for 4–10). On a T4 each epoch takes well under a minute.
The best model (highest validation mIoU) is saved as `my_best_unet.pth`, then Task 4 runs on it.

In [ ]:
%cd /content/lab4
import os
assert os.path.exists('unet_lab.py'), 'unet_lab.py not found - re-run step 2 (mount Drive + unzip) first'
!python unet_lab.py --mode train --epochs 10 --lr 1e-3 --no-show 2>&1 | grep -v "it/s"
# If this cell ends with an error (Traceback), the figures below will be missing - read the error first.

## 4. Show the figures

In [ ]:
import os
from IPython.display import Image, display
OUT = '/content/lab4/outputs'          # absolute path, works whatever the current folder is
print('Files in outputs:', sorted(os.listdir(OUT)) if os.path.isdir(OUT) else 'NO outputs FOLDER')
for f in ['task1_samples.png', 'task3_curves.png', 'task4_predictions.png', 'task4_per_class_iou.txt']:
    path = f'{OUT}/{f}'
    if not os.path.exists(path):
        print(f'MISSING: {f}  -> the training cell (step 3) did not finish; re-run it and check its output')
    elif f.endswith('.png'):
        print(f); display(Image(filename=path, width=900))
    else:
        print(open(path).read())

## 5. Save results back to Google Drive (Colab deletes `/content` when the session ends)

In [ ]:
%cd /content/lab4
!mkdir -p "$DRIVE_DIR/results"
!cp my_best_unet.pth outputs/*.png outputs/*.txt "$DRIVE_DIR/results/"
!ls "$DRIVE_DIR/results"
# or download straight to your PC:
# from google.colab import files; files.download('/content/lab4/my_best_unet.pth')

## 6. Inference only (no training)
`--mode infer` skips Task 3 completely: it builds the U-Net, loads the saved weights, calls `model.eval()` and predicts under `torch.no_grad()`.

In [ ]:
%cd /content/lab4
# your own trained model
!python unet_lab.py --mode infer --ckpt my_best_unet.pth --no-show 2>&1 | grep -v "it/s"
# the provided pretrained model (needs best_unet.pth copied in step 2)
# !python unet_lab.py --mode infer --ckpt best_unet.pth --no-show